In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import random
from scipy.stats import pearsonr
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from collections import defaultdict
import community as community_louvain
from pathlib import Path 
from sklearn.cluster import AgglomerativeClustering 
from sklearn.metrics import silhouette_score
import math
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# 1. Load data
DIRECTORY = Path.cwd().parent / "csv"
nodes_df = pd.read_csv(DIRECTORY / 'nodes.csv')
edges_df = pd.read_csv(DIRECTORY / 'edges.csv')

# 2. Automatically find the Seed DOI from the nodes file
# This makes the code work for any file you upload
seed_nodes = nodes_df[nodes_df['Type'] == 'Seed']['NodeID'].tolist()

if not seed_nodes:
    print("No node with Type='Seed' found.")
else:
    # Focus on the first seed found
    dynamic_seed = seed_nodes[0]
    
    # 3. Build the Graph
    G = nx.from_pandas_edgelist(edges_df,
                                 source='Source',
                                   target='Target',
                                     create_using=nx.DiGraph())

    # 4. Identify the neighbors (targets) for the dynamic seed
    if dynamic_seed in G:
        neighbors = list(G.neighbors(dynamic_seed))
        nodes_to_show = [dynamic_seed] + neighbors
        H = G.subgraph(nodes_to_show)

        # 5. Calculate Layout with high repulsion (k) to avoid overlap
        pos = nx.spring_layout(H, k=2.0, seed=42, scale=5.0)  # Increased scale for more spacing

        # 6. Add Jitter to ensure 100% visibility of overlapping dots
        for node in pos:
            pos[node] = pos[node] + np.random.uniform(-0.04, 0.04, size=2)

        plt.figure(figsize=(10, 8))

        # 7. Draw the focused network
        nx.draw_networkx_edges(H, pos, width=2, edge_color="#1a1b1b", alpha=0.6)
        
        # Color: Seed is Red, others are Blue/Orange
        node_colors = ['#e74c3c' if n == dynamic_seed else '#3498db' for n in H.nodes()]
        nx.draw_networkx_nodes(H, pos, node_size=400, node_color=node_colors)

        # 8. Dynamic Labels: Show 'SEED' and references of seed paper as'P1, P2, P3...'
        label_map = {n: "SEED" if n == dynamic_seed else f"P{i}" 
                     for i, n in enumerate(neighbors, 1)}
        nx.draw_networkx_labels(H, pos, labels=label_map, font_size=10, font_color='white', font_weight='bold')

        plt.title(f"{len(neighbors)} Targets for detected Seed node", fontsize=14)
        plt.axis('off')
        plt.show()
        
        print(f"Verified: Found {len(neighbors)} connections for seed {dynamic_seed}")
        
        # Print total target nodes and total edges
        print(f"Total Target Nodes: {len(neighbors)}")
        print(f"Total Edges: {H.number_of_edges()}")
    else:
        print(f"Seed DOI {dynamic_seed} has no connections in the edges file.")

In [ ]:
# 2. Build the Graph
G = nx.from_pandas_edgelist(edges_df,
                            source='Source',
                             target='Target',
                                create_using=nx.Graph())
node_attr = nodes_df.set_index('NodeID').to_dict('index')
nx.set_node_attributes(G, node_attr)

# 3. Detect Layers Dynamically
# Find the Seed automatically
seeds = [n for n, attr in G.nodes(data=True) if attr.get('Type') == 'Seed']
seed = seeds[0]

# Level 1: Neighbors of the Seed (The 10 Targets)
level1 = list(G.neighbors(seed))

# Level 2: Neighbors of the Targets (The References of Targets)
level2 = set()
for n in level1:
    for neighbor in G.neighbors(n):
        if neighbor != seed and neighbor not in level1:
            level2.add(neighbor)
level2 = list(level2)


# Targets (Level 1) subgraph
level1_nodes = [seed] + level1
H1 = G.subgraph(level1_nodes)

# Level 2 subgraph
level2_nodes = level1_nodes + level2
H2 = G.subgraph(level2_nodes)


print(f"Total Target Nodes (Level 1): {len(level1)}")
print(f"Total Target Edges (Level 1): {H1.number_of_edges()}")
print(f"Total Nodes (Level 2): {len(level2_nodes)}")
print(f"Total Edges (Level 2): {H2.number_of_edges()}")


# 4. Create the Layout
pos = nx.spring_layout(H2, iterations=5000)  # Adjusted k for better spacing in larger graph

# 5. Visualization
plt.figure(figsize=(15, 15))

# Draw everything
nx.draw_networkx_nodes(H2, pos, nodelist=[seed], node_color='#e74c3c', node_size=1000, label='Seed')
nx.draw_networkx_nodes(H2, pos, nodelist=level1, node_color='#3498db', node_size=500, label='Targets')
nx.draw_networkx_nodes(H2, pos, nodelist=level2, node_color="#1b984f", node_size=100, alpha=0.5, label='Level 2')

# Draw connections
nx.draw_networkx_edges(H2, pos, alpha=0.4, edge_color="#1b1b1b")
# Highlight Seed -> Target connections in Red
nx.draw_networkx_edges(H2, pos, edgelist=[(seed, n) for n in level1], width=3, edge_color="#e74c3c", alpha=0.5)

# Label Targets 
labels = {n: f"P{i+1}" for i, n in enumerate(level1)}
labels[seed] = "SEED"
nx.draw_networkx_labels(H2, pos, labels=labels, font_size=10, font_weight='bold')

plt.title(f"Seed → {len(level1)} Targets → {len(level2)} Level 2", fontsize=16)
plt.axis('off')
plt.legend(loc='upper right', fontsize=10)
plt.show()
print(f"Count Check: Seed (1) + Targets ({len(level1)}) + Level 2 ({len(level2)}))")


In [ ]:
# 2. Build the Graph
G = nx.from_pandas_edgelist(edges_df,
                            source='Source',
                             target='Target',
                                create_using=nx.DiGraph())
node_attr = nodes_df.set_index('NodeID').to_dict('index')
nx.set_node_attributes(G, node_attr)

# 3. Detect Layers Dynamically
# Find the Seed automatically
seeds = [n for n, attr in G.nodes(data=True) if attr.get('Type') == 'Seed']
seed = seeds[0]

# Level 1: Neighbors of the Seed (The 10 Targets)
level1 = list(G.neighbors(seed))

# Level 2: Neighbors of the Targets (The References of Targets)
level2 = set()
for n in level1:
    for neighbor in G.neighbors(n):
        if neighbor != seed and neighbor not in level1:
            level2.add(neighbor)
level2 = list(level2)

# Level 3: Neighbors of Level 2 nodes
level3 = set()
for n in level2:
    for neighbor in G.neighbors(n):
        if neighbor != seed and neighbor not in level1 and neighbor not in level2:
            level3.add(neighbor)
level3 = list(level3)

# Targets (Level 1) subgraph
level1_nodes = [seed] + level1
H1 = G.subgraph(level1_nodes)

# Level 2 subgraph
level2_nodes = level1_nodes + level2
H2 = G.subgraph(level2_nodes)

# Level 3 subgraph
level3_nodes = level2_nodes + level3
H3 = G.subgraph(level3_nodes)

print(f"Total Target Nodes (Level 1): {len(level1)}")
print(f"Total Target Edges (Level 1): {H1.number_of_edges()}")
print(f"Total Nodes (Level 2): {len(level2_nodes)}")
print(f"Total Edges (Level 2): {H2.number_of_edges()}")
print(f"Total Nodes (Level 3): {len(level3_nodes)}")
print(f"Total Edges (Level 3): {H3.number_of_edges()}")

# 4. Create the Layout
pos = nx.spring_layout(H3, k=2, scale=1)  # Adjusted k for better spacing in larger graph

for node in pos:
    pos[node] = pos[node] + np.random.uniform(-0.04, 0.04, size=2)

# 5. Visualization
plt.figure(figsize=(15, 15))

# Draw everything
nx.draw_networkx_nodes(G, pos, nodelist=[seed], node_color='#e74c3c', node_size=1000, label='Seed')
nx.draw_networkx_nodes(G, pos, nodelist=level1, node_color='#3498db', node_size=500, label='Targets')
nx.draw_networkx_nodes(G, pos, nodelist=level2, node_color="#1b984f", node_size=100, alpha=0.5, label='Level 2')
nx.draw_networkx_nodes(G, pos, nodelist=level3, node_color="#9b59b6", node_size=50, alpha=0.4, label='Level 3')

# Draw connections
nx.draw_networkx_edges(G, pos, alpha=0.4, edge_color="#1b1b1b")
# Highlight Seed -> Target connections in Red
nx.draw_networkx_edges(G, pos, edgelist=[(seed, n) for n in level1], width=3, edge_color="#e74c3c", alpha=0.5)

# Label the 10 Targets as P1, P2... P10 so you can count them easily
labels = {n: f"P{i+1}" for i, n in enumerate(level1)}
labels[seed] = "SEED"
nx.draw_networkx_labels(G, pos, labels=labels, font_size=10, font_weight='bold')

plt.title(f"Seed → {len(level1)} Targets → {len(level2)} Level 2 → {len(level3)} Level 3", fontsize=16)
plt.axis('off')
plt.legend(loc='upper right', fontsize=10)
plt.show()
print(f"Count Check: Seed (1) + Targets ({len(level1)}) + Level 2 ({len(level2)}) + Level 3 ({len(level3)})")


Katz Centrality for Level 1 (Seed's Neighbors)

In [ ]:
# Create a subgraph containing only seed and level 1 nodes
level1_nodes = [seed] + level1
G_level1_directed = nx.DiGraph()
edges_df_clean = edges_df.dropna(subset=["Source", "Target"])
for edge in edges_df_clean.itertuples(index=False):
    if edge.Source in level1_nodes and edge.Target in level1_nodes:
        G_level1_directed.add_edge(edge.Source, edge.Target)


alpha_level1 = 0.1  # Higher alpha since this is a smaller graph
katz_level1 = nx.katz_centrality_numpy(G_level1_directed, alpha=alpha_level1)

katz_level1_df = pd.DataFrame(katz_level1.items(), columns=["DOI", "Katz_Centrality_Level1"])
katz_level1_df = katz_level1_df.merge(
    nodes_df[['NodeID', 'PaperTitle', 'Abstract']], 
    left_on='DOI', 
    right_on='NodeID', 
    how='left'
).drop(columns=['NodeID'])


katz_level1_df.head(5)

In [ ]:
#Pearson Correlation in Katz Prestige
delete_fractions = [0.01, 0.05, 0.10, 0.20]
num_repeats = 100
katz_orig_rank = pd.Series(katz_level1).rank(method="min")

results = []
for frac in delete_fractions:
    for run in range(num_repeats):
        G_temp = G_level1_directed.copy()
        num_edges = int(frac * G_temp.number_of_edges())
        removed_edges = random.sample(list(G_temp.edges()), num_edges)
        G_temp.remove_edges_from(removed_edges)

        try:
            katz_temp = nx.katz_centrality_numpy(G_temp, alpha= alpha_level1)
            katz_temp_rank = pd.Series(katz_temp).reindex(katz_orig_rank.index).rank(method="min")
            corr, _ = pearsonr(katz_orig_rank, katz_temp_rank)
        except:
            corr = np.nan  

        results.append({
            "Fraction_Removed": frac,
            "Run": run,
            "Pearson_Correlation": corr
        })

results_df = pd.DataFrame(results)
plt.figure(figsize=(8, 5))
results_df.boxplot(column="Pearson_Correlation", by="Fraction_Removed")
plt.title("Robustness of Katz Prestige Centrality - Level 1 Subgraph")
plt.suptitle("")
plt.xlabel("Edge Deletion Fraction")
plt.ylabel("Pearson Correlation")
plt.grid(True)
plt.tight_layout()
plt.show()

Katz Centrality for Level 1 and Level 2 (Seed + Two Hops)

In [ ]:
# Add level 2 nodes and edges
G_level2_directed = G_level1_directed.copy()
level2_nodes = level1_nodes + level2

for edge in edges_df_clean.itertuples(index=False):
    if edge.Source in level2_nodes and edge.Target in level2_nodes:
        if not G_level2_directed.has_edge(edge.Source, edge.Target):
            G_level2_directed.add_edge(edge.Source, edge.Target)

alpha_level2 = 0.05  # Adjusted alpha for medium-sized graph
katz_level2 = nx.katz_centrality_numpy(G_level2_directed,
                                               alpha=alpha_level2)

katz_level2_df = pd.DataFrame(katz_level2.items(),
                                      columns=["DOI", "Katz_Centrality_Level2"])
katz_level2_df = katz_level2_df.merge(
    nodes_df[['NodeID', 'PaperTitle', 'Abstract']], 
    left_on='DOI', 
    right_on='NodeID', 
    how='left'
).drop(columns=['NodeID'])

katz_level2_df.head(5)

In [ ]:
#Pearson Correlation in Katz Prestige - Level 2
delete_fractions = [0.01, 0.05, 0.10, 0.20]
num_repeats = 100
katz_orig_rank = pd.Series(katz_level2).rank(method="min")

results = []
for frac in delete_fractions:
    for run in range(num_repeats):
        G_temp = G_level2_directed.copy()
        num_edges = int(frac * G_temp.number_of_edges())
        removed_edges = random.sample(list(G_temp.edges()), num_edges)
        G_temp.remove_edges_from(removed_edges)

        try:
            katz_temp = nx.katz_centrality_numpy(G_temp, alpha= alpha_level2)
            katz_temp_rank = pd.Series(katz_temp).reindex(katz_orig_rank.index).rank(method="min")
            corr, _ = pearsonr(katz_orig_rank, katz_temp_rank)
        except:
            corr = np.nan  

        results.append({
            "Fraction_Removed": frac,
            "Run": run,
            "Pearson_Correlation": corr
        })

results_df = pd.DataFrame(results)
plt.figure(figsize=(8, 5))
results_df.boxplot(column="Pearson_Correlation", by="Fraction_Removed")
plt.title("Robustness of Katz Prestige Centrality - Level 2 Subgraph")
plt.suptitle("")
plt.xlabel("Edge Deletion Fraction")
plt.ylabel("Pearson Correlation")
plt.grid(True)
plt.tight_layout()
plt.show()

Katz Prestige (or Centrality) Calculation

In [ ]:
# Katz Centrality Calculation
edges_df_cleaned = edges_df.dropna(subset=["Source", "Target"])

#Convert edge list to directed graph
G_original = nx.from_pandas_edgelist(edges_df_cleaned, 
                                     source='Source',
                                     target='Target',
                                     create_using=nx.DiGraph())

alpha = 0.005
katz_centrality = nx.katz_centrality_numpy(G_original, alpha=alpha)

katz_df = pd.DataFrame(katz_centrality.items(), columns=["DOI", "Katz_Prestige"])

katz_df = katz_df.dropna(subset=["DOI"])

katz_df = katz_df.merge(nodes_df[['NodeID', 'PaperTitle', 'Abstract']], left_on='DOI', right_on='NodeID', how='left')
katz_df = katz_df.drop(columns=['NodeID'])

katz_df = katz_df.sort_values(by="Katz_Prestige", ascending=False).reset_index(drop=True)

katz_df.head(5)


Interpretation of Katz Centrality Across All Levels

Pearson correlation coefficient can be used to measure the linear relationship between Katz prestige scores. How katz prestige ranking change when a network slightly altered. It is revealing the robustness of the network.

In [ ]:
# Robustness check for Katz prestige on the full network
delete_fractions = [0.01, 0.05, 0.10, 0.20]
num_repeats = 100
RANDOM_SEED = 42
MAX_ITER = 5000
TOL = 1e-6

rng = random.Random(RANDOM_SEED)
edges_list = list(G_original.edges())
m = len(edges_list)

def safe_katz_centrality(G, alpha=None, max_iter=1000, tol=1e-6, fallback_scc=True):
    if G.number_of_nodes() == 0:
        return {}
    if alpha is None:
        max_out_degree = max((deg for _, deg in G.out_degree()), default=0)
        alpha = 0.9 / max_out_degree if max_out_degree > 0 else 0.01
    try:
        return nx.katz_centrality(G, alpha=alpha, max_iter=max_iter, tol=tol)
    except Exception:
        if not fallback_scc:
            raise
        sccs = list(nx.strongly_connected_components(G))
        if not sccs:
            return {}
        largest_scc = max(sccs, key=len)
        if len(largest_scc) < 3:
            return {}
        G_scc = G.subgraph(largest_scc).copy()
        return nx.katz_centrality(G_scc, alpha=alpha, max_iter=max_iter, tol=tol)

katz_orig = safe_katz_centrality(G_original, alpha=alpha, max_iter=MAX_ITER, tol=TOL, fallback_scc=True)
katz_orig_rank = pd.Series(katz_orig).rank(method="min")

results = []
for frac in delete_fractions:
    num_edges_remove = int(frac * m)
    for run in range(num_repeats):
        G_temp = G_original.copy()
        if num_edges_remove > 0:
            removed_edges = rng.sample(edges_list, num_edges_remove)
            G_temp.remove_edges_from(removed_edges)

        try:
            katz_temp = safe_katz_centrality(G_temp, alpha=alpha, max_iter=MAX_ITER, tol=TOL, fallback_scc=True)
            katz_temp_rank = pd.Series(katz_temp).reindex(katz_orig_rank.index).rank(method="min")
            valid = katz_orig_rank.notna() & katz_temp_rank.notna()
            if valid.sum() >= 2:
                corr, _ = pearsonr(katz_orig_rank[valid], katz_temp_rank[valid])
            else:
                corr = np.nan
        except Exception:
            corr = np.nan
        results.append({
            "Fraction_Removed": frac,
            "Run": run,
            "Pearson_Correlation": corr,
            "Num_Edges_Removed": num_edges_remove
        })

results_df = pd.DataFrame(results)
results_df.head(15)

plt.figure(figsize=(8, 5))
results_df.boxplot(column="Pearson_Correlation", by="Fraction_Removed")
plt.title("Robustness of Katz Prestige Centrality")
plt.suptitle("")
plt.xlabel("Edge Deletion Fraction")
plt.ylabel("Pearson Correlation")
plt.grid(True)
plt.tight_layout()
plt.show()

Eigenvector Centrality Measure

In [ ]:
G_original = nx.from_pandas_edgelist(
    edges_df_cleaned,
    source="Source",
    target="Target",
    create_using=nx.DiGraph()
)


eigen_centrality = nx.eigenvector_centrality(G_original, max_iter=5000, tol=1e-06)


eigen_df = pd.DataFrame(eigen_centrality.items(), columns=["DOI", "Eigenvector_Centrality"])
eigen_df = eigen_df.dropna(subset=["DOI"])
eigen_df = eigen_df.merge(
    nodes_df[["NodeID", "PaperTitle", "Abstract"]],
    left_on="DOI",
    right_on="NodeID",
    how="left"
).drop(columns=["NodeID"])

eigen_df.head(5)

In [ ]:
delete_fractions = [0.01, 0.05, 0.10, 0.20]
num_repeats = 100

MAX_ITER = 5000
TOL = 1e-6
RANDOM_SEED = 42 

rng = random.Random(RANDOM_SEED)

def eigenvector_centrality_safe(G_original, max_iter=5000, tol=1e-6, weight=None, fallback_scc=True):
    try:
        return nx.eigenvector_centrality(G_original, max_iter=max_iter, tol=tol, weight=weight)
    except nx.PowerIterationFailedConvergence:
        if not fallback_scc:
            raise
        sccs = list(nx.strongly_connected_components(G_original))
        if not sccs:
            return {}
        largest_scc = max(sccs, key=len)
        if len(largest_scc) < 3:
            return {}
        G_scc = G_original.subgraph(largest_scc).copy()
        return nx.eigenvector_centrality(G_scc, max_iter=max_iter, tol=tol, weight=weight)
    
eig_orig_rank = pd.Series(eigen_centrality).rank(method="min")
edges_list = list(G_original.edges())
m = len(edges_list)

results = []
for frac in delete_fractions:
    for run in range(num_repeats):
        G_temp = G_original.copy()
        num_edges_remove = int(frac * m)
        removed_edges = random.sample(list(G_temp.edges()), num_edges_remove)
        G_temp.remove_edges_from(removed_edges)

        try:
            eig_temp = eigenvector_centrality_safe(G_temp, max_iter=MAX_ITER, tol=TOL, weight=None, fallback_scc=True)
            eig_temp_rank = pd.Series(eig_temp).reindex(eig_orig_rank.index).rank(method="min")
            valid = eig_orig_rank.notna() & eig_temp_rank.notna()
            if valid.sum() >= 2:
                corr, _ = pearsonr(eig_orig_rank[valid], eig_temp_rank[valid])
            else:
                corr = np.nan

        except Exception:
            corr = np.nan

        results.append({
            "Fraction_Removed": frac,
            "Run": run,
            "Pearson_Correlation": corr,
            "Num_Edges_Removed": num_edges_remove
        })

results_df = pd.DataFrame(results)

plt.figure(figsize=(8, 5))
results_df.boxplot(column="Pearson_Correlation", by="Fraction_Removed")
plt.title("Robustness of Eigenvector Centrality")
plt.suptitle("")
plt.xlabel("Edge Deletion Fraction")
plt.ylabel("Pearson Correlation")
plt.grid(True)

Betweenness Centrality

In [ ]:
#Betweenness Centrality Calculation
edges_df_cleaned = edges_df.dropna(subset=["Source", "Target"])

# Convert edge list to directed graph
G_original = nx.from_pandas_edgelist(
    edges_df_cleaned,
    source="Source",
    target="Target",
    create_using=nx.DiGraph()
)

betweenness_centrality = nx.betweenness_centrality(G_original)

betweenness_df = pd.DataFrame(betweenness_centrality.items(), columns=["DOI", "Betweenness_Centrality"])
betweenness_df = betweenness_df.dropna(subset=["DOI"])
betweenness_df = betweenness_df.merge(
    nodes_df[["NodeID", "PaperTitle", "Abstract"]],
    left_on="DOI",
    right_on="NodeID",
    how="left"
).drop(columns=["NodeID"])

betweenness_df.head(5)

In [ ]:
# Robustness check for betweenness centrality (approx on large networks)
delete_fractions = [0.01, 0.05, 0.10, 0.20]
num_repeats = 100

RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

nodes_list = list(G_original.nodes())
n_nodes = len(nodes_list)

# Limit samples for large graphs to keep runtime manageable
MIN_K = 50
MAX_K = 600
K_SAMPLES = min(MAX_K, max(MIN_K, int(np.sqrt(n_nodes) * 4))) if n_nodes > 0 else 0
sample_nodes = rng.sample(nodes_list, min(K_SAMPLES, n_nodes)) if n_nodes > 0 else []

def safe_approx_betweenness(G, sample_nodes, seed, fallback_scc=True):
    if G.number_of_nodes() == 0:
        return {}
    k = min(len(sample_nodes), G.number_of_nodes())
    if k == 0:
        return {}
    try:
        return nx.betweenness_centrality(G, k=k, normalized=True, endpoints=False, seed=seed)
    except Exception:
        if not fallback_scc:
            return {}
        sccs = list(nx.strongly_connected_components(G))
        if not sccs:
            return {}
        largest_scc = max(sccs, key=len)
        if len(largest_scc) < 3:
            return {}
        G_scc = G.subgraph(largest_scc).copy()
        k_scc = min(k, G_scc.number_of_nodes())
        return nx.betweenness_centrality(G_scc, k=k_scc, normalized=True, endpoints=False, seed=seed)

bet_orig = safe_approx_betweenness(G_original, sample_nodes, RANDOM_SEED, fallback_scc=True)
bet_orig_rank = pd.Series(bet_orig).rank(method="min")
edges_list = list(G_original.edges())
m = len(edges_list)

results = []
for frac in delete_fractions:
    num_edges_remove = int(frac * m)
    for run in range(num_repeats):
        G_temp = G_original.copy()
        if num_edges_remove > 0:
            removed_edges = rng.sample(edges_list, num_edges_remove)
            G_temp.remove_edges_from(removed_edges)

        try:
            bet_temp = safe_approx_betweenness(G_temp, sample_nodes, RANDOM_SEED, fallback_scc=True)
            bet_temp_rank = pd.Series(bet_temp).reindex(bet_orig_rank.index).rank(method="min")
            valid = bet_orig_rank.notna() & bet_temp_rank.notna()
            if valid.sum() >= 2:
                corr, _ = pearsonr(bet_orig_rank[valid], bet_temp_rank[valid])
            else:
                corr = np.nan
        except Exception:
            corr = np.nan

        results.append({
            "Fraction_Removed": frac,
            "Run": run,
            "Pearson_Correlation": corr,
            "Num_Edges_Removed": num_edges_remove,
            "K_SAMPLES": len(sample_nodes)
        })

results_df = pd.DataFrame(results)

plt.figure(figsize=(8, 5))
results_df.boxplot(column="Pearson_Correlation", by="Fraction_Removed")
plt.title("Robustness of Betweenness Centrality")
plt.suptitle("")
plt.xlabel("Edge Deletion Fraction")
plt.ylabel("Pearson Correlation")
plt.grid(True)
plt.tight_layout()
plt.show()

Closeness Centrality

In [ ]:
#Closeness Centrality Calculation
edges_df_cleaned = edges_df.dropna(subset=["Source", "Target"])

# Convert edge list to directed graph
G_original = nx.from_pandas_edgelist(
    edges_df_cleaned,
    source="Source",
    target="Target",
    create_using=nx.DiGraph()
 )

closeness_centrality = nx.closeness_centrality(G_original)

closeness_df = pd.DataFrame(closeness_centrality.items(), columns=["DOI", "Closeness_Centrality"])

closeness_df = closeness_df.dropna(subset=["DOI"])

closeness_df = closeness_df.merge(
    nodes_df[["NodeID", "PaperTitle", "Abstract"]],
    left_on="DOI",
    right_on="NodeID",
    how="left"
 ).drop(columns=["NodeID"])

closeness_df.head(5)

In [ ]:
#Robustness check for Closeness centrality on large networks
delete_fractions = [0.01, 0.05, 0.10, 0.20]
num_repeats = 100


RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

nodes_list = list(G_original.nodes())
n_nodes = len(nodes_list)

# Limit samples for large graphs to keep runtime manageable
MIN_K = 50
MAX_K = 1000
K_SAMPLES = min(MAX_K, max(MIN_K, int(np.sqrt(n_nodes) * 5))) if n_nodes > 0 else 0
sample_nodes = rng.sample(nodes_list, min(K_SAMPLES, n_nodes)) if n_nodes > 0 else []

def approx_closeness(G, sample_nodes):
    if G.number_of_nodes() == 0:
        return {}
    closeness = {}
    for node in sample_nodes:
        try:
            closeness[node] = nx.closeness_centrality(G, u=node)
        except Exception:
            closeness[node] = np.nan
    return closeness

close_orig = approx_closeness(G_original, sample_nodes)
close_orig_rank = pd.Series(close_orig).rank(method="min")

edges_list = list(G_original.edges())
m = len(edges_list)

results = []
for frac in delete_fractions:
    num_edges_remove = int(frac * m)
    for run in range(num_repeats):
        G_temp = G_original.copy()
        if num_edges_remove > 0:
            removed_edges = rng.sample(edges_list, num_edges_remove)
            G_temp.remove_edges_from(removed_edges)

        try:
            close_temp = approx_closeness(G_temp, sample_nodes)
            close_temp_rank = pd.Series(close_temp).reindex(close_orig_rank.index).rank(method="min")
            valid = close_orig_rank.notna() & close_temp_rank.notna()
            if valid.sum() >= 2:
                corr, _ = pearsonr(close_orig_rank[valid], close_temp_rank[valid])
            else:
                corr = np.nan
        except Exception:
            corr = np.nan

        results.append({
            "Fraction_Removed": frac,
            "Run": run,
            "Pearson_Correlation": corr,
            "Num_Edges_Removed": num_edges_remove,
            "K_SAMPLES": len(sample_nodes)
        })

results_df = pd.DataFrame(results)

plt.figure(figsize=(8, 5))
results_df.boxplot(column="Pearson_Correlation", by="Fraction_Removed")
plt.title("Robustness of Closeness Centrality")
plt.suptitle("")
plt.xlabel("Edge Deletion Fraction")
plt.ylabel("Pearson Correlation")
plt.grid(True)
plt.tight_layout()
plt.show()

Lovain Comunity Detection 

In [ ]:
seed_paper = "<DOI_of_the_Seed_Paper>" # Replace with the actual DOI of the seed paper from your nodes.csv

# Compute weights if not present using TF-IDF cosine similarity
if 'weight' not in edges_df.columns:
    # Build a mapping from NodeID to its semantic information
    sem_lookup = nodes_df.set_index('NodeID')['PaperTitle'].fillna('').to_dict()
    node_ids = list(sem_lookup.keys())
    docs = [sem_lookup[nid] for nid in node_ids]
    if len(docs) > 0:
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix = vectorizer.fit_transform(docs)
        node_idx = {nid: idx for idx, nid in enumerate(node_ids)}
        def compute_weight(row):
            i = node_idx.get(row['Source'])
            j = node_idx.get(row['Target'])
            if i is None or j is None:
                return 0.0
            # Cosine similarity is the dot product of normalized TF-IDF vectors
            return float((tfidf_matrix[i] @ tfidf_matrix[j].T).A[0][0])
        edges_df['weight'] = edges_df.apply(compute_weight, axis=1)
    else:
        # Fallback to zero weights
        edges_df['weight'] = 0.0

# Remove edges with zero weight
filtered_edges_df = edges_df[edges_df['weight'] > 0].copy()
# If there are weighted edges, filter by lower quartile to keep most significant relationships
if len(filtered_edges_df) > 0:
    thresh = filtered_edges_df['weight'].quantile(0.25)
    filtered_edges_df = filtered_edges_df[filtered_edges_df['weight'] >= thresh]
else:
    filtered_edges_df = edges_df.copy()

# Build an undirected weighted graph
G = nx.from_pandas_edgelist(
    filtered_edges_df,
    'Source',
    'Target',
    edge_attr=['weight'],
    create_using=nx.Graph()
 )

# Guard against zero total weight (causes Louvain division by zero)
total_weight = sum(data.get('weight', 1.0) for _, _, data in G.edges(data=True))
if G.number_of_edges() == 0 or total_weight == 0:
    # Fall back to unweighted graph on all edges
    G = nx.from_pandas_edgelist(
        edges_df,
        'Source',
        'Target',
        create_using=nx.Graph()
    )
    nx.set_edge_attributes(G, 1.0, "weight")
    total_weight = G.number_of_edges()
    print("Warning: zero total weight; falling back to unweighted graph.")
if G.number_of_edges() == 0:
    raise ValueError("Graph has no edges; cannot run Louvain.")

# Search over different resolution parameters and select the one with highest modularity
resolutions = [0.5, 1.0, 1.5]
best_partition = None
best_modularity = -1
best_res = None
for res in resolutions:
    part = community_louvain.best_partition(G, resolution=res, weight='weight', random_state=42)
    mod = community_louvain.modularity(part, G, weight='weight')
    if mod > best_modularity:
        best_modularity = mod
        best_partition = part
        best_res = res

partition = best_partition
modularity = best_modularity
print(f"Selected resolution: {best_res} with modularity {modularity:.4f}")

# Identify seed paper's community
core_comm = partition.get(seed_paper, "Not Found")
print(f"Seed Paper '{seed_paper}' is in Community: {core_comm}")

# Organise nodes by community
communities = defaultdict(list)
for node, cid in partition.items():
    communities[cid].append(node)

community_df = pd.DataFrame(
    [(cid, paper) for cid, papers in communities.items() for paper in papers],
    columns=["Community", "DOI"]
 )

# Merge with node attributes to retrieve titles and abstracts
community_df = community_df.merge(
    nodes_df[['NodeID', 'PaperTitle', 'Abstract', 'SemanticInformation']],
    left_on='DOI',
    right_on='NodeID',
    how='left'
 ).drop(columns=['NodeID'])

# Aggregate information per community for inspection
community_groups_detailed = community_df.groupby("Community").apply(
    lambda x: x[['DOI', 'PaperTitle', 'Abstract', 'SemanticInformation']].to_dict('records')
).to_dict()

community_groups_df = pd.DataFrame([
    {
        "Community": m,
        "DOIs": "; ".join(str(paper['DOI']) for paper in papers if pd.notna(paper['DOI'])),
        "PaperTitles": "; ".join(str(paper['PaperTitle']) for paper in papers if pd.notna(paper.get('PaperTitle'))),
        "Abstracts": "; ".join(str(paper['Abstract']) for paper in papers if pd.notna(paper.get('Abstract'))),
        "SemanticInformation": "; ".join(str(paper['SemanticInformation']) for paper in papers if pd.notna(paper.get('SemanticInformation')))
    }
    for m, papers in community_groups_detailed.items()
 ])

# Save community-level information to CSV for downstream analysis
community_df.to_csv(DIRECTORY / "community_ids_with_dois.csv", index=False)
community_groups_df.to_csv(DIRECTORY / "community_groups_detailed.csv", index=False)


In [ ]:
# === Plot: Full Network by Community ===
pos = nx.spring_layout(G,  center=(0, 0), seed=21)
base_cmap = plt.get_cmap("tab20")
colors = lambda i: base_cmap(i % base_cmap.N)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 9), gridspec_kw={'width_ratios': [3, 1]})
pos = nx.spring_layout(G, seed=42)
node_colors =  [colors(partition[node]) for node in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=50, ax=ax1)
nx.draw_networkx_edges(G, pos, width=0.5, alpha=0.4, ax=ax1)
ax1.set_title("Louvain Community Detection for Citation Network", fontsize=15)
ax1.axis("off")
ax2.axis("off")
patches = [mpatches.Patch(color=colors(i), label=f"Community {i}") for i in sorted(communities.keys())]
ax2.legend(handles=patches, loc='lower right', fontsize=10, frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# Community Projection Graph 
G_proj = nx.Graph()
for cid, nodes in communities.items():
    G_proj.add_node(cid, size=len(nodes))

for u, v, data in G.edges(data=True):
    cu = partition[u]
    cv = partition[v]
    if cu != cv:
        if G_proj.has_edge(cu, cv):
            G_proj[cu][cv]['weight'] += 1
        else:
            G_proj.add_edge(cu, cv, weight=1)


plt.figure(figsize=(5.5, 6))
proj_pos = nx.spring_layout(G_proj, seed=42, k=5)
proj_colors = [colors(n) for n in G_proj.nodes()]
proj_sizes = [G_proj.nodes[n]['size'] * 10 for n in G_proj.nodes()]
edge_widths = [G_proj[u][v]['weight'] / 10 for u, v in G_proj.edges()]
nx.draw(G_proj, proj_pos, with_labels=True, node_color=proj_colors,
        node_size=proj_sizes, width=edge_widths, edge_color='#1b1b1b', alpha=0.6)
plt.title("Community Projection Graph of Citation Network", fontsize=13)
# plt.savefig("community_projection_graph.png", dpi=300, bbox_inches='tight')
plt.show()


Agglomerative Clustering

In [ ]:
community_df = pd.read_csv(DIRECTORY /"community_ids_with_dois.csv")
G = nx.from_pandas_edgelist(edges_df, "Source", "Target")
seed_paper = "<DOI_of_the_Seed_Paper>" # Replace with the actual DOI of the seed paper from your nodes.csv
partition = {row["DOI"]: row["Community"] for _, row in community_df.iterrows()}

community_nodes = {}
for _, row in community_df.iterrows():
    community_nodes.setdefault(row["Community"], set()).add(row["DOI"])

G_project = nx.Graph()
for c1, nodes1 in community_nodes.items():
    for c2, nodes2 in community_nodes.items():
        if c1 < c2 and any(G.has_edge(n1, n2) for n1 in nodes1 for n2 in nodes2):
            G_project.add_edge(c1, c2)

adjacency_matrix = nx.to_numpy_array(G_project, nodelist=sorted(G_project.nodes()))
distance_matrix = 1 - adjacency_matrix
np.fill_diagonal(distance_matrix, 0)
community_ids = sorted(G_project.nodes())

# Improve domain consistency using PaperTitle + Abstract similarity between communities
stop_words = {
    "the", "and", "for", "with", "from", "that", "this", "these", "those", "using",
    "paper", "study", "method", "methods", "approach", "approaches", "results", "result",
    "based", "show", "shown", "novel", "new", "use", "used", "model", "models"
}
node_text = nodes_df.set_index("NodeID").apply(
    lambda r: " ".join(
        str(v) for v in [r.get("PaperTitle", ""), r.get("Abstract", "")] if pd.notna(v)
    ),
    axis=1
).to_dict()

def text_tokens(text):
    tokens = []
    for t in text.lower().split():
        t = "".join(ch for ch in t if ch.isalpha())
        if len(t) > 2 and t not in stop_words:
            tokens.append(t)
    return set(tokens)

community_nodes_dict = {cid: nodes for cid, nodes in community_nodes.items()}
community_tokens = {}
for cid, nodes in community_nodes_dict.items():
    tokens = set()
    for n in nodes:
        text = node_text.get(n)
        if text:
            tokens |= text_tokens(text)
    if tokens:
        community_tokens[cid] = tokens

for i, c1 in enumerate(community_ids):
    t1 = community_tokens.get(c1)
    if not t1:
        continue
    for j in range(i + 1, len(community_ids)):
        c2 = community_ids[j]
        t2 = community_tokens.get(c2)
        if not t2:
            continue
        union = t1 | t2
        if union:
            overlap = len(t1 & t2) / len(union)
            if overlap >= 0.15:
                distance_matrix[i, j] *= 0.9
                distance_matrix[j, i] *= 0.9

threshold_candidates = np.linspace(0.2, 0.9, 15)
best_threshold = None
best_score = -1

for threshold in threshold_candidates:
    model = AgglomerativeClustering(metric="precomputed", linkage="average",distance_threshold=threshold,n_clusters=None)
    labels = model.fit_predict(distance_matrix)
    if len(set(labels)) < 2 or len(set(labels)) >= len(labels):
        continue
    score = silhouette_score(distance_matrix, labels, metric="precomputed")
    if score > best_score:
        best_score = score
        best_threshold = threshold
     
print(f"Optimal distance threshold: {best_threshold:.3f}")
print(f"Best silhouette score: {best_score:.4f}")

clustering = AgglomerativeClustering(
    metric='precomputed',
    linkage='average',
    distance_threshold=best_threshold, 
    n_clusters=None
)

macro_labels = clustering.fit_predict(distance_matrix)
map_macro_community = {comm: macro for comm, macro in zip(community_ids, macro_labels)}
community_df["Macro_Community"] = community_df["Community"].map(map_macro_community)

seed_paper_macro_comm = community_df.loc[community_df["DOI"] == seed_paper, "Macro_Community"].values
if len(seed_paper_macro_comm) > 0:
    print(f"Seed paper '{seed_paper}' is in Macro Community: {seed_paper_macro_comm[0]}")
else:
    print(f"Seed paper '{seed_paper}' not found in the dataset.")

# Merge with nodes_df to get PaperTitle if not already present
if 'PaperTitle' not in community_df.columns:
    community_df = community_df.merge(nodes_df[['NodeID', 'PaperTitle', 'Abstract','SemanticInformation']], left_on='DOI', right_on='NodeID', how='left')
    community_df = community_df.drop(columns=['NodeID'])

community_df.to_csv(DIRECTORY /"macro_communities.csv", index=False)

# Create macro_groups with PaperTitle
macro_groups_detailed = community_df.groupby("Macro_Community").apply(
    lambda x: x[['DOI', 'PaperTitle', 'Abstract', 'SemanticInformation']].to_dict('records')
).to_dict()

macro_groups_df = pd.DataFrame([
    {
        "Macro_Community": m,
        "DOIs": "; ".join(str(paper['DOI']) for paper in papers if pd.notna(paper['DOI'])),
        "PaperTitles": "; ".join(str(paper['PaperTitle']) for paper in papers if pd.notna(paper.get('PaperTitle'))),
        "Abstracts": "; ".join(str(paper['Abstract']) for paper in papers if pd.notna(paper.get('Abstract'))),
        "SemanticInformation": "; ".join(str(paper['SemanticInformation']) for paper in papers if pd.notna(paper.get('SemanticInformation')))
    }
    for m, papers in macro_groups_detailed.items()
])
macro_groups_df.to_csv(DIRECTORY /"macro_community_groups.csv", index=False)


In [ ]:
# === Color Mapping =====
macro_ids = sorted(set(macro_labels))
cmap = cm.tab20
macro_color_dict = {m: cmap(i % 20) for i, m in enumerate(macro_ids)}
act_macro_map = dict(zip(community_df["DOI"], community_df["Macro_Community"]))

# === Plot: Full Network by Macro-Community ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 10), gridspec_kw={'width_ratios': [3, 1]})
pos = nx.spring_layout(G, seed=42)
node_colors = [macro_color_dict.get(act_macro_map.get(n, -1), "#1b1b1b") for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=50, ax=ax1)
nx.draw_networkx_edges(G, pos, width=0.5, alpha=0.4, ax=ax1)
ax1.set_title("Macro-Community Detection for Citation Network", fontsize=15)
ax1.axis("off")
ax2.axis("off")
patches = [mpatches.Patch(color=macro_color_dict[i], label=f"Macro Community {i}") for i in macro_ids]
ax2.legend(handles=patches, loc='lower right', fontsize=13, frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
macro_nodes = community_df.groupby("Macro_Community")["DOI"].apply(set).to_dict()
G_proj_macro = nx.Graph()

for i, mi in enumerate(macro_ids):
    for mj in macro_ids[i + 1:]:
        edge_count = sum(
            1 for a1 in macro_nodes[mi]
              for a2 in macro_nodes[mj]
              if G.has_edge(a1, a2)
        )
        if edge_count > 0:
            G_proj_macro.add_edge(mi, mj, weight=edge_count)



macro_sizes = [len(macro_nodes[m]) * 2 for m in G_proj_macro.nodes()]
macro_colors = [macro_color_dict[m] for m in G_proj_macro.nodes()]
edge_widths = [G_proj_macro[u][v]['weight'] /10 for u, v in G_proj_macro.edges()]

plt.figure(figsize=(5.5, 6))
pos = nx.spring_layout(G_proj_macro, seed=42, k=8)

nx.draw(
    G_proj_macro,
    pos,
    with_labels=True,
    node_size=macro_sizes,
    node_color=macro_colors,
    width=edge_widths,
    edge_color="#1b1b1b",
    font_size=8
)

plt.title("Community Projection Graph of Citation Network", fontsize=10)
plt.axis("off")
plt.margins(0.15)
plt.tight_layout()
plt.show()

In [ ]:
# Line graph: silhouette score vs number of clusters
num_louvain = len(community_ids)
max_k = num_louvain - 1
if max_k < 2:
    print("Not enough Louvain communities to compute silhouette scores.")
else:
    cluster_range = list(range(2, max_k + 1))
    silhouette_scores = []

    for k in cluster_range:
        model = AgglomerativeClustering(
            metric="precomputed",
            linkage="average",
            n_clusters=k
        )
        labels = model.fit_predict(distance_matrix)
        score = silhouette_score(distance_matrix, labels, metric="precomputed")
        silhouette_scores.append(score)

    plt.figure(figsize=(15, 10))
    plt.plot(cluster_range, silhouette_scores, marker="o", color="#1f77b4")
    plt.title("Silhouette Score vs Number of Clusters", fontsize=11)
    plt.xlabel("Number of Clusters")
    plt.ylabel("Silhouette Score")
    plt.xticks(cluster_range)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()